In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import gradio as gr
import re

# ==== 基本設定 ====
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = './data/f1_model_final_v3.pth'
os.makedirs('./data', exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# ==== 特徵欄位 ====
CAT_COLS = ['Grand Prix', 'Team', 'Driver', 'Nationality']
NUM_COLS = [
    'year', 'Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years',
    'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years',
    'Driver_Prev_Season_FL_Count', 'Is_Home_Race'
]
TARGET_COL = 'is_winner'

# ==== 資料前處理相關 ====
def clean_string(text):
    """去除多餘空白字元"""
    if isinstance(text, str):
        return re.sub(r'\s+', ' ', text).strip()
    return text

def get_country_from_gp(gp_name):
    """根據 GP 名稱對應國家簡碼"""
    GP_COUNTRY_MAP = {
        'British': 'GBR', 'Great Britain': 'GBR', 'Monaco': 'MON', 'Italian': 'ITA', 'Italy': 'ITA',
        'German': 'GER', 'Germany': 'GER', 'Belgian': 'BEL', 'Belgium': 'BEL', 'French': 'FRA', 'France': 'FRA',
        'Dutch': 'NED', 'Spanish': 'ESP', 'Spain': 'ESP', 'Brazilian': 'BRA', 'Brazil': 'BRA',
        'Japanese': 'JPN', 'Japan': 'JPN', 'Canadian': 'CAN', 'Canada': 'CAN', 'Austrian': 'AUT', 'Austria': 'AUT',
        'Hungarian': 'HUN', 'Hungary': 'HUN', 'Mexican': 'MEX', 'Mexico': 'MEX', 'Australian': 'AUS', 'Australia': 'AUS',
        'United States': 'USA', 'USA': 'USA', 'Swiss': 'SUI', 'Switzerland': 'SUI',
    }
    if not isinstance(gp_name, str): return None
    for key, country_code in GP_COUNTRY_MAP.items():
        if key in gp_name: return country_code
    return None

def fill_missing(df, cat_cols, num_cols):
    """補齊缺漏欄位並填入預設值"""
    for col in cat_cols:
        if col not in df.columns: df[col] = 'Unknown'
        df[col] = df[col].fillna('Unknown')
    for col in num_cols:
        if col not in df.columns: df[col] = 0
        df[col] = df[col].fillna(0)
    return df

# ==== 1. 讀取與清理資料 ====
def load_data():
    """讀取所有 CSV 資料、基本清理"""
    d = './data/'
    winners = pd.read_csv(d + 'winners.csv', encoding='utf-8')
    drivers = pd.read_csv(d + 'drivers_updated.csv', encoding='utf-8')
    teams = pd.read_csv(d + 'teams_updated.csv', encoding='utf-8')
    fastest_laps = pd.read_csv(d + 'fastest_laps_updated.csv', encoding='utf-8')
    # 文字欄位清理
    for df in [winners, drivers, teams, fastest_laps]:
        for c in df.select_dtypes(include='object'):
            df[c] = df[c].map(clean_string)
    # 年分
    winners['year'] = pd.to_datetime(winners['Date'], errors='coerce').dt.year.astype(int)
    drivers.rename(columns={'Car': 'Team'}, inplace=True)
    for df in [drivers, teams, fastest_laps]:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype(int)
        if 'Pos' in df.columns: df['Pos'] = pd.to_numeric(df['Pos'], errors='coerce')
    # 排除 Indy 500
    winners = winners[~winners['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    fastest_laps = fastest_laps[~fastest_laps['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    return winners, drivers, teams, fastest_laps

# ==== 2. 特徵工程 ====
def make_features(drivers, teams, fastest_laps, def_pos_drv=50, def_pos_team=20):
    """所有 lag 特徵合併一遍"""
    drivers = drivers.sort_values(['Driver', 'year'])
    drivers['Prev_Year_Driver_PTS'] = drivers.groupby('Driver')['PTS'].shift(1).fillna(0)
    drivers['Prev_Year_Driver_Pos'] = drivers.groupby('Driver')['Pos'].shift(1).fillna(def_pos_drv)
    drivers['Driver_Experience_Years'] = drivers['year'] - drivers.groupby('Driver')['year'].transform('min')
    teams = teams.sort_values(['Team', 'year'])
    teams['Prev_Year_Team_PTS'] = teams.groupby('Team')['PTS'].shift(1).fillna(0)
    teams['Prev_Year_Team_Pos'] = teams.groupby('Team')['Pos'].shift(1).fillna(def_pos_team)
    teams['Team_Experience_Years'] = teams['year'] - teams.groupby('Team')['year'].transform('min')
    drivers = drivers.merge(
        teams[['Team', 'year', 'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years']],
        on=['Team', 'year'], how='left').fillna(0)
    fl = fastest_laps.groupby(['year', 'Driver']).size().reset_index(name='FL_Count')
    fl['Driver_Prev_Season_FL_Count'] = fl.groupby('Driver')['FL_Count'].shift(1).fillna(0)
    drivers = drivers.merge(fl[['Driver', 'year', 'Driver_Prev_Season_FL_Count']],
                           on=['Driver', 'year'], how='left').fillna(0)
    return drivers

# ==== 3. 組成建模資料 ====
def build_model_df(winners, drivers, def_pos_drv=50, def_pos_team=20):
    """每場比賽所有 driver 預測資料"""
    data = []
    drivers_by_year = {y: g for y, g in drivers.groupby('year')}
    for _, race in winners.iterrows():
        y, gp, winner = race['year'], race['Grand Prix'], race['Winner']
        race_country = get_country_from_gp(gp)
        if y not in drivers_by_year: continue
        for _, d in drivers_by_year[y].iterrows():
            is_home = 1 if race_country and d['Nationality'] == race_country else 0
            data.append({
                'year': y, 'Grand Prix': gp, 'Driver': d['Driver'], 'Team': d['Team'], 'Nationality': d['Nationality'],
                'Prev_Year_Driver_PTS': d['Prev_Year_Driver_PTS'], 'Prev_Year_Driver_Pos': d['Prev_Year_Driver_Pos'],
                'Driver_Experience_Years': d['Driver_Experience_Years'],
                'Prev_Year_Team_PTS': d['Prev_Year_Team_PTS'], 'Prev_Year_Team_Pos': d['Prev_Year_Team_Pos'],
                'Team_Experience_Years': d['Team_Experience_Years'],
                'Driver_Prev_Season_FL_Count': d['Driver_Prev_Season_FL_Count'],
                'Is_Home_Race': is_home,
                'is_winner': int(d['Driver'] == winner)
            })
    return pd.DataFrame(data)

# ==== 4. 編碼 + 標準化 ====
def encode_and_scale(train, test, cat_cols, num_cols):
    encoders, cat_dims = {}, {}
    for c in cat_cols:
        le = LabelEncoder()
        train[c] = le.fit_transform(train[c].astype(str))
        test[c] = test[c].map(lambda x: le.transform([x])[0] if x in le.classes_ else len(le.classes_))
        encoders[c] = le
        cat_dims[c] = len(le.classes_) + 1
    scaler = StandardScaler()
    train[num_cols] = scaler.fit_transform(train[num_cols])
    test[num_cols] = scaler.transform(test[num_cols])
    return train, test, encoders, scaler, cat_dims

# ==== 5. Dataset ====
class F1Dataset(Dataset):
    def __init__(self, df, cat_cols, num_cols, target_col):
        self.x_cat = df[cat_cols].values
        self.x_num = df[num_cols].values
        self.y = df[target_col].values
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.x_cat[idx], dtype=torch.long), \
               torch.tensor(self.x_num[idx], dtype=torch.float32), \
               torch.tensor(self.y[idx], dtype=torch.float32)

# ==== 6. Model ====
class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num_feats, emb_dim=32, hidden_dim=128, dropout_rate=0.4):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(dim, emb_dim) for dim in cat_dims])
        self.bn_num = nn.BatchNorm1d(num_num_feats)
        self.fc = nn.Sequential(
            nn.Linear(len(cat_dims)*emb_dim + num_num_feats, hidden_dim), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim//2, 1)
        )
    def forward(self, x_cat, x_num):
        x = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], 1)
        x = torch.cat([x, self.bn_num(x_num)], 1)
        return self.fc(x)

# ==== 7. 訓練 ====
def train_model(model, trainloader, testloader, n_epoch=30, lr=5e-4, patience=7, model_path=MODEL_PATH):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    best_test_loss = float('inf'); no_improve = 0
    train_losses, test_losses = [], []
    for epoch in range(n_epoch):
        model.train(); train_loss = 0
        for x_cat, x_num, y_true in trainloader:
            x_cat, x_num, y_true = x_cat.to(DEVICE), x_num.to(DEVICE), y_true.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_cat, x_num), y_true.unsqueeze(1))
            loss.backward(); optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(trainloader)
        train_losses.append(avg_train_loss)
        # 驗證
        model.eval(); test_loss = 0
        with torch.no_grad():
            for x_cat, x_num, y_true in testloader:
                x_cat, x_num, y_true = x_cat.to(DEVICE), x_num.to(DEVICE), y_true.to(DEVICE)
                loss = criterion(model(x_cat, x_num), y_true.unsqueeze(1))
                test_loss += loss.item()
        avg_test_loss = test_loss / len(testloader)
        test_losses.append(avg_test_loss)
        print(f"Epoch {epoch+1} | Train: {avg_train_loss:.4f} | Test: {avg_test_loss:.4f}")
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            torch.save(model.state_dict(), model_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping."); break
    import matplotlib.pyplot as plt
    plt.figure(); plt.plot(train_losses, label='Train'); plt.plot(test_losses, label='Test')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
    plt.savefig('./data/loss_curve_final.png'); plt.close()

# ==== 8. 評估 ====
def evaluate_model(model, dataloader, model_path=MODEL_PATH):
    if model_path and os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval(); y_true, y_pred = [], []
    with torch.no_grad():
        for x_cat, x_num, y in dataloader:
            x_cat, x_num, y = x_cat.to(DEVICE), x_num.to(DEVICE), y.to(DEVICE)
            probs = torch.sigmoid(model(x_cat, x_num))
            pred = (probs > 0.5).squeeze().int()
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy() if pred.ndim > 0 else [pred.item()])
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=['Not Winner','Winner'], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ==== 9. Gradio預測 ====
GRADIO_VARS = dict(
    label_encoders={}, scaler=None, model=None, driver_info_by_year={},
    all_years=[], all_gps=[], cat_cols_ordered=[], num_cols_ordered=[]
)

def gradio_predict_winner_probabilities(year_input, grand_prix_input):
    gv = GRADIO_VARS
    if not gv["model"] or not gv["label_encoders"] or (gv["num_cols_ordered"] and gv["scaler"] is None):
        return "Error: Model or preprocessors not loaded."
    try: year = int(year_input)
    except: return "Error: Invalid year."
    if not grand_prix_input: return "Error: Grand Prix input empty."
    drivers = gv["driver_info_by_year"].get(year, [])
    if not drivers: return f"No driver data for {year}."
    model = gv["model"]; model.eval(); res = []
    for p in drivers:
        cat = [gv["label_encoders"][col].transform([str(p.get(col, 'Unknown'))])[0] 
               if str(p.get(col, '')) in set(gv["label_encoders"][col].classes_)
               else len(gv["label_encoders"][col].classes_) for col in gv["cat_cols_ordered"]]
        num = [float(p.get(col, 0)) if col != 'Is_Home_Race' else
               1.0 if get_country_from_gp(grand_prix_input) == p.get('Nationality') else 0.0
               for col in gv["num_cols_ordered"]]
        # 避免警告：用 DataFrame 傳給 scaler
        x_num_df = pd.DataFrame([num], columns=gv["num_cols_ordered"])
        x_num = torch.tensor(gv["scaler"].transform(x_num_df), dtype=torch.float32).to(DEVICE)
        x_cat = torch.tensor([cat], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            prob = torch.sigmoid(model(x_cat, x_num)).cpu().item()
        res.append((p.get('Driver','N/A'), p.get('Team','N/A'), prob))
    res.sort(key=lambda x:x[2], reverse=True)
    # 只顯示前五名
    output = f"Predictions for {grand_prix_input}, {year}:\n"
    for i, (driver, team, prob) in enumerate(res[:5], 1):
        output += f"{i}. {driver} ({team}): {prob:.2%}\n"
    return output.strip()

# ==== 10. 主流程 ====
if __name__ == '__main__':
    # 讀資料
    winners, drivers, teams, fastest_laps = load_data()
    def_pos_drv = int(drivers['Pos'].max(skipna=True))+5 if 'Pos' in drivers else 50
    def_pos_team = int(teams['Pos'].max(skipna=True))+5 if 'Pos' in teams else 20
    drivers_feat = make_features(drivers, teams, fastest_laps, def_pos_drv, def_pos_team)
    modeling_df = build_model_df(winners, drivers_feat, def_pos_drv, def_pos_team)
    modeling_df['race_id'] = modeling_df['year'].astype(str) + "_" + modeling_df['Grand Prix'].astype(str)
    unique_race_ids = modeling_df['race_id'].unique()
    # train/test 分割
    if len(unique_race_ids) < 2:
        train_df, test_df = modeling_df.copy(), modeling_df.copy()
    else:
        train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=SEED, shuffle=True)
        train_df = modeling_df[modeling_df['race_id'].isin(train_ids)].copy()
        test_df = modeling_df[modeling_df['race_id'].isin(test_ids)].copy()
    train_df = fill_missing(train_df, CAT_COLS, NUM_COLS)
    test_df = fill_missing(test_df, CAT_COLS, NUM_COLS)
    for df_ in [train_df, test_df]:
        if 'race_id' in df_.columns: df_.drop(columns=['race_id'], inplace=True)
    train_df, test_df, GRADIO_VARS["label_encoders"], GRADIO_VARS["scaler"], cat_dims_map = \
        encode_and_scale(train_df, test_df, CAT_COLS, NUM_COLS)
    GRADIO_VARS["cat_cols_ordered"] = CAT_COLS
    GRADIO_VARS["num_cols_ordered"] = NUM_COLS
    # DataLoader
    train_set = F1Dataset(train_df, CAT_COLS, NUM_COLS, TARGET_COL)
    test_set = F1Dataset(test_df, CAT_COLS, NUM_COLS, TARGET_COL)
    weights = [1./(train_df[TARGET_COL].value_counts().get(t,1)+1e-6) for t in train_df[TARGET_COL]]
    sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights))
    train_loader = DataLoader(train_set, batch_size=256, sampler=sampler)
    test_loader = DataLoader(test_set, batch_size=256, shuffle=False)
    # Model
    ordered_cat_dims = [cat_dims_map[c] for c in CAT_COLS]
    num_num_feats = len(NUM_COLS)
    GRADIO_VARS["model"] = F1DNN(ordered_cat_dims, num_num_feats).to(DEVICE)
    TRAIN_FLAG = not os.path.exists(MODEL_PATH)
    if TRAIN_FLAG:
        train_model(GRADIO_VARS["model"], train_loader, test_loader, n_epoch=50, patience=10)
    evaluate_model(GRADIO_VARS["model"], test_loader, model_path=MODEL_PATH if not TRAIN_FLAG else None)
    # Gradio 下拉選單
    for y, group in drivers_feat.groupby('year'): GRADIO_VARS["driver_info_by_year"][y] = group.to_dict('records')
    GRADIO_VARS["all_years"] = sorted(list(drivers_feat['year'].unique()))
    GRADIO_VARS["all_gps"] = sorted(list(winners['Grand Prix'].unique()))
    # Gradio 介面
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# F1 Grand Prix Winner Predictor (Top 5)")
        with gr.Row():
            year_dd = gr.Dropdown(label="Year", choices=GRADIO_VARS["all_years"], value=GRADIO_VARS["all_years"][-1])
            gp_dd = gr.Dropdown(label="Grand Prix", choices=GRADIO_VARS["all_gps"], value=GRADIO_VARS["all_gps"][0])
        predict_btn = gr.Button("Predict Probabilities")
        output_tb = gr.Textbox(label="Predicted Top 5", lines=8, interactive=False)
        predict_btn.click(gradio_predict_winner_probabilities, inputs=[year_dd, gp_dd], outputs=[output_tb])
        with gr.Accordion("Training Information", open=False):
            if os.path.exists("./data/loss_curve_final.png"):
                gr.Image(value="./data/loss_curve_final.png", label="Loss Curve")
            else:
                gr.Markdown("Loss curve image not found.")
    demo.launch(share=False)


Epoch 1 | Train: 0.5643 | Test: 0.4551
Epoch 2 | Train: 0.4057 | Test: 0.4025
Epoch 3 | Train: 0.3453 | Test: 0.3940
Epoch 4 | Train: 0.3266 | Test: 0.3618
Epoch 5 | Train: 0.3081 | Test: 0.3817
Epoch 6 | Train: 0.2908 | Test: 0.3567
Epoch 7 | Train: 0.2769 | Test: 0.3579
Epoch 8 | Train: 0.2610 | Test: 0.3405
Epoch 9 | Train: 0.2490 | Test: 0.3369
Epoch 10 | Train: 0.2434 | Test: 0.3193
Epoch 11 | Train: 0.2287 | Test: 0.3435
Epoch 12 | Train: 0.2252 | Test: 0.3230
Epoch 13 | Train: 0.2169 | Test: 0.3116
Epoch 14 | Train: 0.2054 | Test: 0.3295
Epoch 15 | Train: 0.2050 | Test: 0.3237
Epoch 16 | Train: 0.1936 | Test: 0.3198
Epoch 17 | Train: 0.1888 | Test: 0.3235
Epoch 18 | Train: 0.1806 | Test: 0.3144
Epoch 19 | Train: 0.1777 | Test: 0.3221
Epoch 20 | Train: 0.1675 | Test: 0.3308
Epoch 21 | Train: 0.1617 | Test: 0.3443
Epoch 22 | Train: 0.1686 | Test: 0.3237
Epoch 23 | Train: 0.1631 | Test: 0.3349
Early stopping.
Accuracy: 0.8908722109533469
              precision    recall  f1-score 